# YouTube Trending — Gold Business Cases
### Gold Datamarts → Actionable Business Insights

**Purpose:** This notebook reads the **gold-layer datamarts** and derives scenario-specific analytical tables that answer targeted business questions.

**Upstream Gold Tables (source):**

| Gold Source | Business Case(s) |
|---|---|
| `category_performance` | Scenario 1 — Category Scorecard |
| `tag_trends` | Scenario 2 — Tag Strategy |
| `trending_velocity` | Scenario 3 — Viral Breakout Analysis |
| `weekend_vs_weekday` | Scenario 4 — Publishing Day Strategy |
| `channel_consistency` + `channel_daily_leaderboard` | Scenario 5 — Channel Power Rankings |
| `channel_daily_leaderboard` | Scenario 6 — Leaderboard Monopoly Detection |

**Business Case Tables Produced:**

| # | Output Table | Description |
|---|---|---|
| 1 | `bc_category_scorecard` | Category health: sentiment, size tier, engagement grade |
| 2 | `bc_tag_strategy` | Tag tiers with engagement efficiency & recommendations |
| 3 | `bc_viral_breakouts` | Top 100 viral videos with peak-timing classification |
| 4 | `bc_publishing_strategy` | Day-of-week ranking with composite publishing score |
| 5 | `bc_channel_power_rankings` | Composite power score from consistency + leaderboard presence |
| 6 | `bc_leaderboard_monopoly` | Trending page diversity & channel dominance detection |

---
_Upstream: [gold](#) · Downstream: Dashboards & BI tools_

In [0]:
# ── PySpark imports ──
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# ── Medallion architecture paths (ADLS Gen2) ──
root_path = "abfss://employee@dataanlysisazuredatalake.dfs.core.windows.net"
gold_path = f"{root_path}/gold"

# ── Unity Catalog references ──
gold_sch = "gold_youtube"
youtube_db = "employeedatacatalog"

In [0]:
# ── Load the 6 gold datamarts produced by the gold notebook ──

category_perf   = spark.table(f"{youtube_db}.{gold_sch}.category_performance")         # 16 categories
tag_trends      = spark.table(f"{youtube_db}.{gold_sch}.tag_trends")                    # 51K+ tags
trending_vel    = spark.table(f"{youtube_db}.{gold_sch}.trending_velocity")              # 5.6K videos
weekend_weekday = spark.table(f"{youtube_db}.{gold_sch}.weekend_vs_weekday")             # 7 rows
channel_consist = spark.table(f"{youtube_db}.{gold_sch}.channel_consistency")            # 2.2K channels
channel_leader  = spark.table(f"{youtube_db}.{gold_sch}.channel_daily_leaderboard")     # 2K rows

print("Gold tables loaded successfully")

---
## Scenario 1 — Category Growth & Sentiment Scorecard
**Business Question:** How healthy is each content category? Which categories are growing, well-liked, and highly engaged — and which are controversial or stagnant?

**Source:** `category_performance`  
**Output:** `bc_category_scorecard`

In [0]:
# ============================================================
# BUSINESS CASE 1: bc_category_scorecard
# ============================================================
# Assigns each category a sentiment label, size tier, engagement
# grade, and controversy index. Produces a one-row-per-category
# executive scorecard.

# Step 1 — Classify sentiment from like ratio
df_scorecard = (
    category_perf
    .withColumn("sentiment",
        when(col("avg_like_ratio") >= 95, lit("Excellent"))
        .when(col("avg_like_ratio") >= 90, lit("Good"))
        .when(col("avg_like_ratio") >= 80, lit("Mixed"))
        .otherwise(lit("Controversial")))
)

# Step 2 — Size tier based on total views
df_scorecard = (
    df_scorecard
    .withColumn("size_tier",
        when(col("total_views") >= 10_000_000_000, lit("Mega"))
        .when(col("total_views") >= 3_000_000_000, lit("Large"))
        .when(col("total_views") >= 1_000_000_000, lit("Mid"))
        .otherwise(lit("Niche")))
)

# Step 3 — Engagement grade (A–D)
df_scorecard = (
    df_scorecard
    .withColumn("engagement_grade",
        when(col("avg_engagement_rate") >= 5, lit("A"))
        .when(col("avg_engagement_rate") >= 4, lit("B"))
        .when(col("avg_engagement_rate") >= 3, lit("C"))
        .otherwise(lit("D")))
)

# Step 4 — Controversy index (normalized dislike ratio 0–100)
df_scorecard = (
    df_scorecard
    .withColumn("controversy_index", round(col("dislike_ratio") * 5, 2))  # Scale 0–100
    .withColumn("_bc_ingested_at", current_timestamp())
    .select(
        "category_name", "total_trending_entries", "unique_videos",
        "unique_channels", "total_views", "avg_views",
        "avg_engagement_rate", "avg_like_ratio", "avg_days_to_trend",
        "dislike_ratio", "sentiment", "size_tier", "engagement_grade",
        "controversy_index", "_bc_ingested_at"
    )
    .orderBy(desc("total_views"))
)

# Persist
df_scorecard.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/bc_category_scorecard")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{gold_sch}.bc_category_scorecard
              USING DELTA LOCATION '{gold_path}/bc_category_scorecard'""")

print(f"Categories scored: {df_scorecard.count()}")
df_scorecard.limit(10).display()

---
## Scenario 2 — Tag Strategy Recommendations
**Business Question:** Which tags deliver the best ROI in engagement per view? Which are oversaturated, and which are hidden gems?

**Source:** `tag_trends`  
**Output:** `bc_tag_strategy`

In [0]:
# ============================================================
# BUSINESS CASE 2: bc_tag_strategy
# ============================================================
# Classifies tags into tiers by popularity, computes an engagement-
# efficiency metric, and labels each tag with a recommendation.

# Step 1 — Tag tier by popularity rank
df_tags = (
    tag_trends
    .withColumn("tag_tier",
        when(col("tag_popularity_rank") <= 50, lit("Mega"))         # Top 50 tags
        .when(col("tag_popularity_rank") <= 200, lit("Power"))      # 51–200
        .when(col("tag_popularity_rank") <= 1000, lit("Niche"))     # 201–1000
        .otherwise(lit("Long Tail")))                               # 1000+
)

# Step 2 — Engagement efficiency = engagement per log(appearances)
df_tags = (
    df_tags
    .withColumn("engagement_efficiency",
        round(col("avg_engagement_rate") / log(col("trending_appearances") + 1), 3))
)

# Step 3 — Recommendation label
df_tags = (
    df_tags
    .withColumn("recommendation",
        when((col("tag_tier") == "Mega") & (col("avg_engagement_rate") >= 4), lit("Must Use"))
        .when((col("tag_tier") == "Power") & (col("avg_engagement_rate") >= 3), lit("Recommended"))
        .when((col("tag_tier") == "Niche") & (col("engagement_efficiency") >= 1), lit("Hidden Gem"))
        .when((col("tag_tier") == "Mega") & (col("avg_engagement_rate") < 3), lit("Saturated"))
        .otherwise(lit("Low Priority")))
    .withColumn("_bc_ingested_at", current_timestamp())
    .select(
        "tag_name", "tag_popularity_rank", "tag_tier",
        "trending_appearances", "unique_videos", "unique_channels",
        "total_views", "avg_views_per_entry", "avg_engagement_rate",
        "avg_like_ratio", "engagement_efficiency", "recommendation",
        "_bc_ingested_at"
    )
    .orderBy("tag_popularity_rank")
)

# Persist
df_tags.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/bc_tag_strategy")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{gold_sch}.bc_tag_strategy
              USING DELTA LOCATION '{gold_path}/bc_tag_strategy'""")

print(f"Tags classified: {df_tags.count():,}")
print("\nRecommendation breakdown:")
df_tags.groupBy("recommendation").agg(count("*").alias("count")).orderBy(desc("count")).display()
df_tags.limit(10).display()

---
## Scenario 3 — Viral Breakout Analysis
**Business Question:** Which videos exploded the fastest? Do they peak early (day 1–2) or build momentum over time?

**Source:** `trending_velocity`  
**Output:** `bc_viral_breakouts`

In [0]:
# ============================================================
# BUSINESS CASE 3: bc_viral_breakouts
# ============================================================
# Filters to top 100 most viral videos by virality_rank.
# Classifies each by peak-day timing and computes a sustainability
# score (how long they stayed vs when they peaked).

# Step 1 — Filter to top 100 viral videos
df_viral = trending_vel.filter(col("virality_rank") <= 100)

# Step 2 — Classify peak timing
df_viral = (
    df_viral
    .withColumn("peak_timing",
        when(col("peak_day_num") <= 3, lit("Early Peaker"))         # Peaked in first 3 days
        .when(col("peak_day_num") <= 7, lit("Mid Peaker"))          # 4–7 days
        .otherwise(lit("Late Bloomer")))                            # 8+ days
)

# Step 3 — Sustainability score = trending days / peak day
df_viral = (
    df_viral
    .withColumn("sustainability_score",
        round(col("total_trending_days") / nullif(col("peak_day_num"), lit(0)), 2))
    .withColumn("views_per_trending_day",
        round(col("peak_views") / col("total_trending_days"), 0))
    .withColumn("_bc_ingested_at", current_timestamp())
    .select(
        "virality_rank", "video_id", "total_trending_days",
        "peak_views", "max_daily_view_gain", "avg_daily_view_gain",
        "avg_daily_growth_pct", "avg_engagement_rate", "avg_like_ratio",
        "peak_day_num", "peak_date", "peak_timing",
        "sustainability_score", "views_per_trending_day", "_bc_ingested_at"
    )
    .orderBy("virality_rank")
)

# Persist
df_viral.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/bc_viral_breakouts")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{gold_sch}.bc_viral_breakouts
              USING DELTA LOCATION '{gold_path}/bc_viral_breakouts'""")

print(f"Top 100 viral videos")
print("\nPeak timing breakdown:")
df_viral.groupBy("peak_timing").agg(count("*").alias("count"), round(avg("peak_views"), 0).alias("avg_peak_views")).display()
df_viral.limit(10).display()

---
## Scenario 4 — Optimal Publishing Day Strategy
**Business Question:** Which day of the week is best for publishing content that will trend? Are weekends better than weekdays?

**Source:** `weekend_vs_weekday`  
**Output:** `bc_publishing_strategy`

In [0]:
# ============================================================
# BUSINESS CASE 4: bc_publishing_strategy
# ============================================================
# Ranks each day of the week by a composite score combining
# total views, engagement rate, and speed-to-trend.
# Produces a recommendation for content scheduling.

# Step 1 — Normalize metrics to 0–100 scale for scoring
max_views = weekend_weekday.agg(max("total_views")).collect()[0][0]
max_eng   = weekend_weekday.agg(max("avg_engagement_rate")).collect()[0][0]
min_dtt   = weekend_weekday.agg(min("avg_days_to_trend")).collect()[0][0]  # Lower is better
max_dtt   = weekend_weekday.agg(max("avg_days_to_trend")).collect()[0][0]

df_publish = (
    weekend_weekday
    .withColumn("view_score",
        round(col("total_views") / lit(max_views) * 100, 1))
    .withColumn("engagement_score",
        round(col("avg_engagement_rate") / lit(max_eng) * 100, 1))
    .withColumn("speed_score",                                     # Inverted: lower days_to_trend = higher score
        round((1 - (col("avg_days_to_trend") - lit(min_dtt)) / nullif(lit(max_dtt - min_dtt), lit(0))) * 100, 1))
)

# Step 2 — Composite publishing score (weighted)
df_publish = (
    df_publish
    .withColumn("composite_score",
        round(col("view_score") * 0.4 + col("engagement_score") * 0.35 + col("speed_score") * 0.25, 1))
    .withColumn("day_rank",
        rank().over(Window.orderBy(desc("composite_score"))))
    .withColumn("recommendation",
        when(col("day_rank") <= 2, lit("Best Days"))
        .when(col("day_rank") <= 5, lit("Good"))
        .otherwise(lit("Avoid")))
    .withColumn("_bc_ingested_at", current_timestamp())
    .select(
        "day_rank", "day_name", "day_type", "is_weekend",
        "trending_entries", "unique_videos", "total_views", "avg_views",
        "avg_engagement_rate", "avg_like_ratio", "avg_days_to_trend",
        "view_score", "engagement_score", "speed_score",
        "composite_score", "recommendation", "_bc_ingested_at"
    )
    .orderBy("day_rank")
)

# Persist
df_publish.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/bc_publishing_strategy")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{gold_sch}.bc_publishing_strategy
              USING DELTA LOCATION '{gold_path}/bc_publishing_strategy'""")

df_publish.display()

---
## Scenario 5 — Channel Power Rankings
**Business Question:** Which channels are both consistent AND dominate the daily leaderboard? True powerhouses vs flash-in-the-pan.

**Source:** `channel_consistency` + `channel_daily_leaderboard`  
**Output:** `bc_channel_power_rankings`

In [0]:
# ============================================================
# BUSINESS CASE 5: bc_channel_power_rankings
# ============================================================
# Combines consistency tier with leaderboard presence to produce
# a composite power score. Channels that are consistent AND
# frequently in the top 10 are the true powerhouses.

# Step 1 — Count leaderboard appearances & top-3 finishes
df_leader_stats = (
    channel_leader
    .groupBy("channel_key", "channel_title")
    .agg(
        count("*").alias("leaderboard_appearances"),
        sum(when(col("daily_rank") <= 3, 1).otherwise(0)).alias("top3_finishes"),
        round(avg("daily_total_views"), 0).alias("avg_leaderboard_views"),
        round(avg("daily_rank"), 1).alias("avg_rank")
    )
)

# Step 2 — Total leaderboard days for percentage calculation
total_leader_days = channel_leader.select("trending_date").distinct().count()

# Step 3 — Join with consistency data and compute power score
df_power = (
    channel_consist.alias("c")
    .join(df_leader_stats.alias("l"),
          col("c.channel_key") == col("l.channel_key"), "left")
    .withColumn("leaderboard_pct",                                 # % of days in top 10
        round(coalesce(col("l.leaderboard_appearances"), lit(0)) / lit(total_leader_days) * 100, 2))
    .withColumn("top3_pct",                                        # % of days in top 3
        round(coalesce(col("l.top3_finishes"), lit(0)) / lit(total_leader_days) * 100, 2))
)

# Step 4 — Composite power score: consistency (40%) + leaderboard (30%) + engagement (30%)
df_power = (
    df_power
    .withColumn("power_score",
        round(
            col("c.consistency_score") * 0.4 +
            col("leaderboard_pct") * 0.3 +
            coalesce(col("c.avg_engagement_rate"), lit(0)) * 6 * 0.3,  # Scale engagement to ~0–100
        2))
    .withColumn("power_rank",
        rank().over(Window.orderBy(desc("power_score"))))
    .withColumn("power_tier",
        when(col("power_rank") <= 10, lit("Elite"))
        .when(col("power_rank") <= 50, lit("Strong"))
        .when(col("power_rank") <= 200, lit("Solid"))
        .otherwise(lit("Emerging")))
    .withColumn("_bc_ingested_at", current_timestamp())
    .select(
        "power_rank", col("c.channel_key"), col("c.channel_title"),
        col("c.channel_tier"), col("c.consistency_score"),
        col("c.trending_days_count"), col("c.lifetime_views"),
        col("c.avg_engagement_rate"),
        "leaderboard_pct", "top3_pct", "avg_leaderboard_views",
        "avg_rank", "power_score", "power_tier", "_bc_ingested_at"
    )
    .orderBy("power_rank")
)

# Persist
df_power.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/bc_channel_power_rankings")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{gold_sch}.bc_channel_power_rankings
              USING DELTA LOCATION '{gold_path}/bc_channel_power_rankings'""")

print(f"Channels ranked: {df_power.count():,}")
print("\nPower tier breakdown:")
df_power.groupBy("power_tier").agg(count("*").alias("channels")).orderBy(desc("channels")).display()
df_power.limit(10).display()

---
## Scenario 6 — Leaderboard Monopoly Detection
**Business Question:** Is the trending page diverse, or do a few channels monopolize it? Which days had the least diversity?

**Source:** `channel_daily_leaderboard`  
**Output:** `bc_leaderboard_monopoly`

In [0]:
# ============================================================
# BUSINESS CASE 6: bc_leaderboard_monopoly
# ============================================================
# Measures trending page diversity per day. Identifies dates
# where a small number of channels dominated and flags
# channels that repeatedly hold top positions.

# Step 1 — Per-day diversity metrics
df_daily_diversity = (
    channel_leader
    .groupBy("trending_date")
    .agg(
        countDistinct("channel_key").alias("unique_channels_in_top10"),
        count("*").alias("total_entries"),
        round(avg("daily_total_views"), 0).alias("avg_views_top10"),
        max("daily_total_views").alias("top_channel_views"),
        min("daily_total_views").alias("bottom_channel_views")
    )
)

# Step 2 — Diversity score and monopoly flag
df_daily_diversity = (
    df_daily_diversity
    .withColumn("diversity_score",                                 # 10 unique = 100%, 1 = monopoly
        round(col("unique_channels_in_top10") / lit(10) * 100, 1))
    .withColumn("view_concentration",                              # Top vs bottom gap
        round(col("top_channel_views") / nullif(col("bottom_channel_views"), lit(0)), 2))
    .withColumn("monopoly_flag",
        when(col("unique_channels_in_top10") <= 5, lit("High Monopoly"))
        .when(col("unique_channels_in_top10") <= 7, lit("Moderate"))
        .otherwise(lit("Diverse")))
    .withColumn("_bc_ingested_at", current_timestamp())
    .orderBy("diversity_score")
)

# Persist
df_daily_diversity.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{gold_path}/bc_leaderboard_monopoly")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {youtube_db}.{gold_sch}.bc_leaderboard_monopoly
              USING DELTA LOCATION '{gold_path}/bc_leaderboard_monopoly'""")

print(f"Days analyzed: {df_daily_diversity.count():,}")
print("\nMonopoly breakdown:")
df_daily_diversity.groupBy("monopoly_flag").agg(
    count("*").alias("days"),
    round(avg("unique_channels_in_top10"), 1).alias("avg_unique_channels")
).orderBy("avg_unique_channels").display()
df_daily_diversity.limit(10).display()